# **Project Name** - Brain Tumor MRI Image Classification

##### **Project Type**    - Classification (Deep Learning / Computer Vision / Medical Imaging)
##### **Contribution**    - Individual
##### **Team Member 1 -** Arunav Kumar

# **Project Summary -**

**Goal.** Classify brain MRI slices into four classes (glioma, meningioma, pituitary tumor, no tumor) with a custom CNN built from scratch and with ImageNet-pretrained transfer-learning models (MobileNetV2, EfficientNetB0, ResNet50V2), then deploy the best model in a Streamlit app.

**Data.** Roboflow "Labeled MRI Brain Tumor Dataset v1": 2,443 JPG images (640x640 RGB) delivered as train / valid / test folders.

**Key data finding (data leakage).** The images are Roboflow exports, so several augmented variants of the *same* source scan sit in different splits. By matching the source ID in the file name, a large share of the provided test/valid images share a source scan with the training set. A model evaluated on that split would look better than it really is. This notebook therefore merges all images and re-splits them *grouped by source scan* (StratifiedGroupKFold, ~64/16/20), so no scan is ever seen in more than one split.

**Method.** Pixel values are normalised inside each model; light augmentation (flip, rotation, zoom, shift, contrast, brightness) is applied only while training; class weights handle the mild imbalance. The custom CNN uses four conv blocks with BatchNormalization and Dropout. Transfer-learning models are trained in two phases: frozen backbone (new head only), then fine-tuning of the top backbone layers at a low learning rate. EarlyStopping, ModelCheckpoint and ReduceLROnPlateau are used for every model.

**Evaluation.** Accuracy, macro precision / recall / F1, per-class recall, confusion matrices and training curves on the untouched test split. Because a missed tumor is the costliest error in this domain, the notebook also reports the percentage of tumor images that were predicted as "no tumor".

**Results.** <<FILL AFTER RUNNING: best model, its test accuracy and macro-F1, custom CNN vs transfer learning gap, tumor-miss rate>>

**Deployment.** The best model is saved as `.h5` and served by a Streamlit app (`app.py`) that takes an uploaded MRI image and shows the predicted class with confidence scores.

> Note: the summary has to be expanded to 500-600 words with the real numbers before submission.

# **GitHub Link -**

<<paste your public GitHub repository link here>>

# **Problem Statement**

**Develop a deep-learning solution that classifies brain MRI images by tumor type. Build a custom CNN from scratch, improve on it with transfer learning using pretrained models, compare them fairly, and deploy the best model in a Streamlit web app that predicts the tumor type and confidence for an uploaded MRI image.**

Business use cases: AI-assisted diagnosis, early detection and patient triage, research / clinical-trial cohort building, and second-opinion tools for under-resourced regions. This project is a learning exercise and **not** a medical device.

# ***Let's Begin !***

**How to run:** set `DATA_ROOT` in the config cell (folder that contains the three zips, or the extracted `train/valid/test` folders). Run once with `QUICK_TEST = True` to dry-run the whole notebook in a few minutes, then set it to `False` for the real run. Use a GPU runtime (Kaggle / Colab).

## ***1. Setup***
### Import Libraries

In [1]:
# Import Libraries
import os, re, glob, json, time, random, zipfile, hashlib, shutil, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus if gpus else "none (training will be slow on CPU)")
for g in gpus:                                    # avoid grabbing all GPU memory at once
    tf.config.experimental.set_memory_growth(g, True)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ------------------------- CONFIG -------------------------
SEED = 42
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]   # folder names
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}

IMG_SIZE = (224, 224)
BATCH_SIZE = 32            # use 16 if you hit out-of-memory on a small GPU
EPOCHS_CNN = 40            # max epochs, EarlyStopping will stop earlier
EPOCHS_HEAD = 12           # transfer learning phase 1 (frozen backbone)
EPOCHS_FINETUNE = 15       # transfer learning phase 2 (top layers unfrozen)
UNFREEZE_LAST = 30         # how many backbone layers to unfreeze in phase 2
LR_CNN, LR_HEAD, LR_FINETUNE = 1e-3, 1e-3, 1e-5

WEIGHTS = "imagenet"       # pretrained weights for transfer learning
QUICK_TEST = False         # True = tiny dry run to check that every cell works

# Where the data is: a folder holding the 3 zip files, OR the extracted train/valid/test folders.
DATA_ROOT = "/kaggle/input/datasets/arunavkumar/brain-tumor-mri"       

if os.path.exists("/kaggle/working"):
    WORK_DIR, OUT_DIR = "/kaggle/working/data", "/kaggle/working/outputs"
elif os.path.exists("/content"):
    WORK_DIR, OUT_DIR = "/content/data_extracted", "/content/outputs"
else:
    WORK_DIR, OUT_DIR = "./data_extracted", "./outputs"
os.makedirs(WORK_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

if QUICK_TEST:
    EPOCHS_CNN, EPOCHS_HEAD, EPOCHS_FINETUNE = 1, 1, 1
    print("QUICK_TEST is ON: tiny subset, 1 epoch per phase. Results are meaningless.")

keras.utils.set_random_seed(SEED)

## ***2. Know Your Data***
### Dataset Loading
If the data is zipped it is extracted first. Every image becomes one row in a DataFrame (path, class, original split, and a **source ID** parsed from the file name, which we need to detect leakage).

In [ ]:
# Extract zips if present, then index every image.
def prepare_data(root):
    zips = sorted(glob.glob(os.path.join(root, "*.zip")))
    if not zips:
        return root                                   # already extracted folders
    for z in zips:
        target = os.path.join(WORK_DIR, Path(z).stem)
        if not os.path.isdir(target):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)
    return WORK_DIR

data_dir = prepare_data(DATA_ROOT)

rows = []
for p in glob.glob(os.path.join(data_dir, "**", "*.jpg"), recursive=True):
    parts = Path(p).parts
    cls = parts[-2]
    if cls not in CLASS_TO_ID:
        continue                                      # ignore anything that is not a class folder
    orig = next((x for x in parts if x in ("train", "valid", "test")), "unknown")
    # Roboflow names files like  Tr-gl_0516_jpg.rf.<hash>.jpg  -> source scan = Tr-gl_0516
    source_id = re.sub(r"_jpg\.rf\..*$", "", os.path.basename(p))
    rows.append((p, cls, CLASS_TO_ID[cls], orig, source_id))

df = pd.DataFrame(rows, columns=["path", "label", "label_id", "orig_split", "source_id"])
df = df.sort_values("path").reset_index(drop=True)
assert len(df) > 0, "No images found - check DATA_ROOT"

if QUICK_TEST:                                        # small balanced subset for a dry run
    df = df.groupby("label", group_keys=False).sample(n=45, random_state=SEED).reset_index(drop=True)
    IMG_SIZE = (96, 96)

print("Images found:", len(df))
df.head()

### Dataset Rows & Columns, Info, Duplicates, Missing values

In [ ]:
print("Rows, columns:", df.shape)
df.info()
print("\nMissing values per column:\n", df.isnull().sum())

# exact duplicate files (same bytes)
hashes = df["path"].apply(lambda p: hashlib.md5(open(p, "rb").read()).hexdigest())
print("\nExact duplicate files:", int(hashes.duplicated().sum()))

# image resolution / colour-mode consistency
sizes, modes = Counter(), Counter()
for p in df["path"]:
    with Image.open(p) as im:
        sizes[im.size] += 1; modes[im.mode] += 1
print("Resolutions:", dict(sizes))
print("Colour modes:", dict(modes))

### Chart 1 - Class distribution in the *original* provided splits

In [ ]:
# Chart 1: class balance per original split
order = ["train", "valid", "test"]
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.countplot(data=df, x="label", hue="orig_split", hue_order=[o for o in order if o in df.orig_split.unique()],
              order=CLASS_NAMES, ax=ax)
ax.set_title("Images per class in the original train / valid / test folders")
ax.set_xlabel("Class"); ax.set_ylabel("Number of images")
for c in ax.containers: ax.bar_label(c, fontsize=8)
plt.tight_layout(); plt.show()

print((df.groupby(["orig_split", "label"]).size().unstack(fill_value=0)))

##### 1. Why did you pick the specific chart?
A grouped bar chart shows class counts and compares splits side by side, which is the quickest way to spot class imbalance and splits with a different class mix.

##### 2. What is/are the insight(s) found from the chart?
Glioma is the largest class and no_tumor the smallest in every split, so the imbalance is mild (roughly 1.7 : 1), not severe. The three splits have a similar class mix.

##### 3. Will the gained insights help creating a positive business impact?
Yes. A mild imbalance means plain accuracy can hide weakness on the smaller classes, so we use class weights and judge models by macro-F1 and per-class recall. Ignoring this could produce a model that looks good overall but misses the minority class (e.g. no_tumor vs tumor confusion), which is a negative-impact risk in a clinical setting.

### Chart 2 - Sample MRI images per class

In [ ]:
# Chart 2: 5 random images per class
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(14, 11))
rng = np.random.default_rng(SEED)
for r, cls in enumerate(CLASS_NAMES):
    sub = df[df.label == cls]
    picks = sub.iloc[rng.choice(len(sub), size=min(5, len(sub)), replace=False)]
    for c, (_, row) in enumerate(picks.iterrows()):
        axes[r, c].imshow(Image.open(row.path).convert("RGB"))
        axes[r, c].axis("off")
        if c == 0: axes[r, c].set_title(cls, loc="left", fontsize=12, fontweight="bold")
plt.suptitle("Sample images per class", y=1.0); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?
An image grid is the only way to actually look at the data: scan orientation, zoom, contrast and artefacts cannot be seen from numbers.

##### 2. What is/are the insight(s) found from the chart?
<<FILL AFTER LOOKING: e.g. different scan planes (axial / sagittal / coronal), different zoom levels and contrast, tumors of varying size and position>>

##### 3. Will the gained insights help creating a positive business impact?
Visible variation in plane, zoom and contrast tells us that augmentation (rotation, zoom, shift, contrast) is justified and that the model must be robust to acquisition differences between hospitals.

### Chart 3 - Is image brightness a shortcut for the class?
If mean brightness differs strongly by class, a model could "cheat" on exposure instead of learning the tumor.

In [ ]:
# Chart 3: mean grey level per image, by class
def mean_gray(p):
    with Image.open(p) as im:
        return np.asarray(im.convert("L").resize((64, 64)), dtype=np.float32).mean()
df["mean_gray"] = df["path"].apply(mean_gray)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=df, x="label", y="mean_gray", order=CLASS_NAMES, ax=ax)
ax.set_title("Mean pixel intensity by class"); ax.set_xlabel("Class"); ax.set_ylabel("Mean grey level (0-255)")
plt.tight_layout(); plt.show()
print(df.groupby("label")["mean_gray"].describe()[["mean", "std", "min", "max"]].round(1))

##### 1. Why did you pick the specific chart?
A box plot compares the distribution of a numeric feature across categories and shows spread and outliers, not just the average.

##### 2. What is/are the insight(s) found from the chart?
<<FILL AFTER RUNNING: do the class boxes overlap heavily or is one class clearly brighter/darker?>>

##### 3. Will the gained insights help creating a positive business impact?
If the classes overlap heavily, brightness is not an easy shortcut and the model must learn real structure. If one class is clearly separated, brightness/contrast augmentation becomes more important so the model does not rely on exposure.

## ***3. Data Leakage Check (important)***
The provided splits come from a Roboflow export. Files that share the same source ID are variants of one scan. If such variants land in both train and test, the test score is inflated. Let's measure it.

In [ ]:
# How many source scans appear in more than one of the provided splits?
src_splits = df.groupby("source_id")["orig_split"].nunique()
print("Unique source scans :", df["source_id"].nunique(), "of", len(df), "images")
print("Scans in >1 split   :", int((src_splits > 1).sum()))

split_sets = {s: set(df.loc[df.orig_split == s, "source_id"]) for s in ["train", "valid", "test"] if (df.orig_split == s).any()}
leak_rows = []
for s in ["valid", "test"]:
    if s not in split_sets: continue
    others = set().union(*[v for k, v in split_sets.items() if k != s]) if s == "test" else split_sets.get("train", set())
    n = int((df.orig_split == s).sum())
    leaked = int(df.loc[(df.orig_split == s) & df.source_id.isin(others), "path"].count())
    leak_rows.append({"split": s, "images": n, "share a source scan with other split(s)": leaked, "percent": round(100 * leaked / n, 1)})
leak_df = pd.DataFrame(leak_rows)
display(leak_df)

# a source ID must belong to exactly one class, otherwise grouping would be unsafe
assert (df.groupby("source_id")["label"].nunique() == 1).all()

##### What did you find, and what do we do about it?
A noticeable share of the provided valid and test images come from scans that are also in the training set. Scores on that test folder would be optimistic. **Fix:** ignore the provided split, and create our own split grouped by `source_id`, so every scan (with all its augmented variants) lives in exactly one of train / validation / test.

In [ ]:
# Leakage-free split: ~64% train / 16% validation / 20% test, stratified by class, grouped by source scan.
sgkf_outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(sgkf_outer.split(df, df["label_id"], df["source_id"]))
df_trainval, df_test = df.iloc[trainval_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

sgkf_inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, val_idx = next(sgkf_inner.split(df_trainval, df_trainval["label_id"], df_trainval["source_id"]))
df_train, df_val = df_trainval.iloc[tr_idx].reset_index(drop=True), df_trainval.iloc[val_idx].reset_index(drop=True)

# hard guarantee: no source scan is shared between any two splits
s_tr, s_va, s_te = set(df_train.source_id), set(df_val.source_id), set(df_test.source_id)
assert not (s_tr & s_va) and not (s_tr & s_te) and not (s_va & s_te), "Leakage between splits!"

split_summary = pd.concat({"train": df_train.label.value_counts(), "val": df_val.label.value_counts(),
                           "test": df_test.label.value_counts()}, axis=1).loc[CLASS_NAMES]
split_summary.loc["TOTAL"] = split_summary.sum()
display(split_summary)
print("No overlap of source scans between train / val / test: OK")

### What data-wrangling did we do?
We merged the three provided splits, parsed a source ID from each file name, and rebuilt the splits grouped by source scan with class stratification. No images were removed (no missing values, no exact duplicates, uniform 640x640 RGB). Resizing and normalisation are done in the input pipeline (next section).

## ***4. Preprocessing & Augmentation***
- **Resize** to 224x224 inside the `tf.data` pipeline (images are cached after resizing so epochs are fast).
- **Normalise**: the custom CNN scales to 0-1 (`Rescaling(1/255)`); each pretrained model gets the scaling *it* was trained with (MobileNetV2 / ResNet50V2: -1..1, EfficientNetB0: handled internally). The scaling layer lives **inside the model**, so the saved `.h5` and the Streamlit app need no extra preprocessing.
- **Augmentation**: horizontal flip, small rotation, zoom, shift and contrast (layers inside the model, active only during training) plus random brightness (applied in the `tf.data` pipeline on training images only, because the Keras `RandomBrightness` layer cannot be saved to `.h5`). Vertical flip is deliberately left out: upside-down brain scans never occur in practice, so it would teach the model unrealistic images.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_resize(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, IMG_SIZE)                  # float32, 0-255
    return tf.cast(tf.round(img), tf.uint8), label        # uint8 keeps the cache small

def random_brightness(x, y):
    # brightness augmentation lives in the data pipeline (training only): the Keras RandomBrightness
    # layer cannot be saved to .h5, which is the required model format
    x = tf.cast(x, tf.float32)
    return tf.clip_by_value(tf.image.random_brightness(x, 25.0), 0.0, 255.0), y

def to_float(x, y):
    return tf.cast(x, tf.float32), y

def make_dataset(frame, training):
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["label_id"].values.astype("int32")))
    ds = ds.map(load_and_resize, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(random_brightness, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    else:
        ds = ds.batch(BATCH_SIZE).map(to_float, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

ds_train = make_dataset(df_train, training=True)
ds_val   = make_dataset(df_val,   training=False)
ds_test  = make_dataset(df_test,  training=False)       # not shuffled, so order matches df_test
y_test = df_test["label_id"].values

# Augmentation block inside the model (only active when training=True); brightness is done in the tf.data pipeline above
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.10),
], name="augmentation")

# Class weights (mild imbalance)
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=df_train["label_id"].values)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})

In [ ]:
# Visualise augmentation: one training image, 7 random augmentations
img_batch, _ = next(iter(ds_val))
sample = img_batch[0:1]
fig, axes = plt.subplots(1, 8, figsize=(18, 2.8))
axes[0].imshow(sample[0].numpy().astype("uint8")); axes[0].set_title("original"); axes[0].axis("off")
for i in range(1, 8):
    aug = augment(sample, training=True)
    aug = tf.clip_by_value(tf.image.random_brightness(aug[0], 25.0), 0, 255).numpy().astype("uint8")
    axes[i].imshow(aug); axes[i].set_title(f"aug {i}"); axes[i].axis("off")
plt.tight_layout(); plt.show()

## ***5. Model Building***
### Shared helpers (callbacks, training, plotting, evaluation)

In [ ]:
def get_callbacks(ckpt_path, threshold=None, patience=6):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True,
                                        initial_value_threshold=threshold, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ]

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def merge_histories(*hists):
    out = {}
    for h in hists:
        for k, v in h.history.items():
            out.setdefault(k, []).extend(v)
    return out

def plot_history(hist, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(hist["accuracy"], label="train"); ax[0].plot(hist["val_accuracy"], label="validation")
    ax[0].set_title(f"{title} - accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(hist["loss"], label="train"); ax[1].plot(hist["val_loss"], label="validation")
    ax[1].set_title(f"{title} - loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

results, histories = {}, {}

def evaluate_model(name, ckpt_path):
    # Reload the best checkpoint from disk and score it on the untouched test split.
    model = keras.models.load_model(ckpt_path)
    t0 = time.time(); probs = model.predict(ds_test, verbose=0); dt = time.time() - t0
    y_pred = probs.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    per_class_recall = precision_recall_fscore_support(y_test, y_pred, labels=np.arange(NUM_CLASSES), zero_division=0)[1]
    tumor_mask = y_test != CLASS_TO_ID["no_tumor"]
    tumor_miss = float(np.mean(y_pred[tumor_mask] == CLASS_TO_ID["no_tumor"]) * 100)   # tumor predicted as "no tumor"
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred), "Macro Precision": p, "Macro Recall": r, "Macro F1": f1,
        "Tumor missed as No-Tumor (%)": tumor_miss,
        "Params (M)": model.count_params() / 1e6,
        "Size (MB)": os.path.getsize(ckpt_path) / 1e6,
        "Inference (ms/img)": 1000 * dt / len(y_test),
        "_y_pred": y_pred, "_probs": probs, "_recall": per_class_recall, "_ckpt": ckpt_path,
    }
    print(f"\n===== {name} : test results =====")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))
    return model

### Model 1 - Custom CNN (from scratch)
Four convolution blocks (two 3x3 convs + BatchNorm + ReLU, then max-pool and Dropout), global average pooling, a dense layer with Dropout, and a softmax output. BatchNormalization stabilises training, Dropout fights overfitting on this small dataset.

In [ ]:
def build_custom_cnn():
    inp = keras.Input(shape=(*IMG_SIZE, 3))
    x = augment(inp)
    x = layers.Rescaling(1.0 / 255)(x)                            # 0-1 normalisation
    for filters in (32, 64, 128, 256):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="custom_cnn")

cnn = build_custom_cnn()
compile_model(cnn, LR_CNN)
cnn.summary()

In [ ]:
CKPT_CNN = os.path.join(OUT_DIR, "custom_cnn.h5")
h = cnn.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_CNN, class_weight=class_weight,
            callbacks=get_callbacks(CKPT_CNN, patience=8), verbose=2)
histories["Custom CNN"] = merge_histories(h)
plot_history(histories["Custom CNN"], "Custom CNN")
evaluate_model("Custom CNN", CKPT_CNN);

### Models 2-4 - Transfer learning (MobileNetV2, EfficientNetB0, ResNet50V2)
Each backbone is loaded with ImageNet weights and its top removed. We add global average pooling, Dropout, a dense layer and a new 4-class softmax head.

- **Phase 1:** backbone frozen, only the new head is trained (learning rate 1e-3).
- **Phase 2:** the top `UNFREEZE_LAST` backbone layers are unfrozen (BatchNorm layers stay frozen, which is the standard safe practice) and trained at a very low learning rate (1e-5) so pretrained features are adapted, not destroyed.

ResNet50V2 is used instead of the original ResNet50 because its input scaling (-1..1) can be expressed with a plain `Rescaling` layer, which keeps the saved `.h5` loadable anywhere without custom code.

In [ ]:
TL_SPECS = {
    "MobileNetV2":    {"fn": keras.applications.MobileNetV2,    "scale": "minus1_1"},
    "EfficientNetB0": {"fn": keras.applications.EfficientNetB0, "scale": None},   # rescales internally (expects 0-255)
    "ResNet50V2":     {"fn": keras.applications.ResNet50V2,     "scale": "minus1_1"},
}

def build_transfer_model(name):
    spec = TL_SPECS[name]
    base = spec["fn"](include_top=False, weights=WEIGHTS, input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    inp = keras.Input(shape=(*IMG_SIZE, 3))
    x = augment(inp)
    if spec["scale"] == "minus1_1":
        x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
    x = base(x, training=False)                                   # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name=name.lower()), base

def train_transfer_model(name):
    model, base = build_transfer_model(name)
    ckpt = os.path.join(OUT_DIR, f"{name.lower()}.h5")

    # Phase 1: train the new head only
    compile_model(model, LR_HEAD)
    h1 = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_HEAD, class_weight=class_weight,
                   callbacks=get_callbacks(ckpt, patience=4), verbose=2)
    best_val_phase1 = min(h1.history["val_loss"])

    # Phase 2: fine-tune top layers of the backbone (BatchNorm stays frozen)
    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAST]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, LR_FINETUNE)                              # must recompile after changing trainable
    h2 = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_FINETUNE, class_weight=class_weight,
                   callbacks=get_callbacks(ckpt, threshold=best_val_phase1, patience=5), verbose=2)

    histories[name] = merge_histories(h1, h2)
    print(f"{name}: phase 1 ran {len(h1.history['loss'])} epochs, phase 2 ran {len(h2.history['loss'])} epochs")
    plot_history(histories[name], name)
    return ckpt

In [ ]:
# Model 2: MobileNetV2
ckpt = train_transfer_model("MobileNetV2")
evaluate_model("MobileNetV2", ckpt);

In [ ]:
# Model 3: EfficientNetB0
ckpt = train_transfer_model("EfficientNetB0")
evaluate_model("EfficientNetB0", ckpt);

In [ ]:
# Model 4: ResNet50V2  (heaviest model - set BATCH_SIZE = 16 above if your GPU runs out of memory)
ckpt = train_transfer_model("ResNet50V2")
evaluate_model("ResNet50V2", ckpt);

## ***6. Model Evaluation & Comparison***
All numbers below are on the **test split** (never used for training, validation, early stopping or model selection), and the test split shares no source scan with training.

In [ ]:
metric_cols = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Tumor missed as No-Tumor (%)",
               "Params (M)", "Size (MB)", "Inference (ms/img)"]
comparison = pd.DataFrame({k: {c: v[c] for c in metric_cols} for k, v in results.items()}).T
comparison = comparison.sort_values("Macro F1", ascending=False)
display(comparison.style.format({"Accuracy": "{:.3f}", "Macro Precision": "{:.3f}", "Macro Recall": "{:.3f}",
                                 "Macro F1": "{:.3f}", "Tumor missed as No-Tumor (%)": "{:.1f}",
                                 "Params (M)": "{:.2f}", "Size (MB)": "{:.1f}", "Inference (ms/img)": "{:.1f}"})
        .background_gradient(subset=["Macro F1"], cmap="Greens"))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
comparison[["Accuracy", "Macro F1"]].plot(kind="bar", ax=ax[0], rot=15)
ax[0].set_title("Test accuracy and macro-F1"); ax[0].set_ylim(0, 1.05)
for c in ax[0].containers: ax[0].bar_label(c, fmt="%.2f", fontsize=8)
recall_df = pd.DataFrame({k: v["_recall"] for k, v in results.items()}, index=CLASS_NAMES).T.loc[comparison.index]
sns.heatmap(recall_df, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax[1])
ax[1].set_title("Per-class recall (share of each class correctly found)")
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrices for every model
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4.5))
axes = np.atleast_1d(axes)
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r["_y_pred"], labels=np.arange(NUM_CLASSES))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Training curves of all models in one view
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in histories.items():
    axes[0].plot(h["val_accuracy"], label=name); axes[1].plot(h["val_loss"], label=name)
axes[0].set_title("Validation accuracy per epoch"); axes[1].set_title("Validation loss per epoch")
for a in axes: a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

### Which metrics matter and which model do we choose?
- **Macro-F1** is the main selection metric: it treats every class equally, so the smaller classes are not drowned out by the larger ones.
- **Per-class recall** shows how many real cases of each class are found.
- **Tumor missed as No-Tumor (%)** is the clinically most dangerous error (a real tumor that the model calls healthy). Between two models with similar macro-F1, prefer the one that misses fewer tumors.
- **Size and inference time** matter for deployment (the brief asks for the most *efficient* model too). A 100 MB `.h5` also exceeds GitHub's normal file limit, so a smaller model is easier to ship.

In [ ]:
# Automatic summary of the comparison (copy the conclusions into your report)
best_name = comparison.index[0]
b = comparison.loc[best_name]
print(f"Best model by macro-F1: {best_name}  (accuracy {b['Accuracy']:.3f}, macro-F1 {b['Macro F1']:.3f}, "
      f"tumor missed as no-tumor {b['Tumor missed as No-Tumor (%)']:.1f}%, size {b['Size (MB)']:.1f} MB)")
cnn_f1 = comparison.loc["Custom CNN", "Macro F1"]
tl = comparison.drop(index="Custom CNN")
print(f"Custom CNN macro-F1: {cnn_f1:.3f} | best transfer-learning macro-F1: {tl['Macro F1'].max():.3f} "
      f"({tl['Macro F1'].idxmax()}) | gap: {tl['Macro F1'].max() - cnn_f1:+.3f}")
print(f"Lowest tumor-miss rate: {comparison['Tumor missed as No-Tumor (%)'].idxmin()} "
      f"({comparison['Tumor missed as No-Tumor (%)'].min():.1f}%)")
print(f"Smallest model: {comparison['Size (MB)'].idxmin()} ({comparison['Size (MB)'].min():.1f} MB)")
weakest = recall_df.loc[best_name].idxmin()
print(f"Weakest class for {best_name}: {weakest} (recall {recall_df.loc[best_name].min():.2f})")

##### Final model choice and why
<<FILL AFTER RUNNING: name the model you deploy and justify with macro-F1, tumor-miss rate, size / speed. Mention the weakest class and which classes get confused in the confusion matrix. Be honest about limits: small dataset, one source, no external validation.>>

## ***7. Save the best model & sanity check***

In [ ]:
# Save the winning model in .h5 format (+ class names) for the Streamlit app
best_ckpt = results[best_name]["_ckpt"]
final_path = os.path.join(OUT_DIR, "best_model.h5")
shutil.copyfile(best_ckpt, final_path)
with open(os.path.join(OUT_DIR, "class_names.json"), "w") as f:
    json.dump(CLASS_NAMES, f)
print("Saved:", final_path, f"({os.path.getsize(final_path)/1e6:.1f} MB)")

# Sanity check: reload from disk, predict a few unseen test images, compare with the earlier predictions
reloaded = keras.models.load_model(final_path)
paths = df_test["path"].values[:12]
batch = np.stack([np.asarray(Image.open(p).convert("RGB").resize(IMG_SIZE[::-1]), dtype=np.float32) for p in paths])
pred_ids = reloaded.predict(batch, verbose=0).argmax(1)
for p, t, q in zip(paths, y_test[:12], pred_ids):
    print(f"{os.path.basename(p)[:28]:30s} true={CLASS_NAMES[t]:11s} pred={CLASS_NAMES[q]:11s} {'OK' if t == q else 'WRONG'}")

## ***8. Streamlit App (separate file)***
The app lives in `app.py` (included in the project folder). Copy `best_model.h5` and `class_names.json` into a `models/` folder next to it, then run:

```
pip install -r requirements.txt
streamlit run app.py
```
It lets the user upload an MRI image, shows the predicted tumor type, the confidence, and the probability of every class.

## ***9. Limitations & Future Work***
- Only ~2.4k images, from a single public source; there is no external hospital validation, so real-world performance is unknown.
- Re-splitting by source scan removed leakage between *our* splits, but we cannot rule out that different Roboflow variants of the same patient exist under different IDs.
- Ideas: Grad-CAM heat-maps to show *where* the model looks, k-fold cross-validation for more stable estimates, test-time augmentation, and calibrating confidence scores before any clinical use.

# **Conclusion**
<<FILL AFTER RUNNING: 4-6 sentences: what was built, final test accuracy / macro-F1 of the chosen model, transfer learning vs custom CNN, the data-leakage finding and its effect, the main limitation.>>

### ***Hurrah! You have successfully completed your Deep Learning Capstone Project !!!***